# 面试题：Trace 怎样做分层 Grading？

分层 grader 将权限、计划、参数、结果验证、回答与副作用分别判定；安全层硬失败，不能用漂亮回答抵消。它不同于 trace 记录：这里要用 fixture 与权威状态对每层作出可复放判决。

## 真实案例

六条退款 trace 分别出现无授权、缺依赖、金额错、未回读、虚假完成和完整成功。

## 基线

基线仅按最终回答是否含完成打分。

## 结果解读

手写 grader 输出层级矩阵和硬门禁。

## 失败案例

无授权调用即使最后说对了也不能通过。

In [1]:
traces = [{'id':'G1','auth':False,'plan':True,'param':True,'verify':True,'answer':True,'effect':False}, {'id':'G2','auth':True,'plan':False,'param':True,'verify':True,'answer':True,'effect':False}, {'id':'G3','auth':True,'plan':True,'param':False,'verify':True,'answer':False,'effect':False}, {'id':'G4','auth':True,'plan':True,'param':True,'verify':False,'answer':True,'effect':False}, {'id':'G5','auth':True,'plan':True,'param':True,'verify':True,'answer':True,'effect':True}, {'id':'G6','auth':True,'plan':True,'param':True,'verify':True,'answer':True,'effect':True}]  # 构造六条具备各层证据的退款 trace。
print('Trace 输入:', traces)  # 输出各层布尔证据。
print('教学说明：effect 表示权威退款账本与当前请求匹配。')  # 说明最终副作用来源。

Trace 输入: [{'id': 'G1', 'auth': False, 'plan': True, 'param': True, 'verify': True, 'answer': True, 'effect': False}, {'id': 'G2', 'auth': True, 'plan': False, 'param': True, 'verify': True, 'answer': True, 'effect': False}, {'id': 'G3', 'auth': True, 'plan': True, 'param': False, 'verify': True, 'answer': False, 'effect': False}, {'id': 'G4', 'auth': True, 'plan': True, 'param': True, 'verify': False, 'answer': True, 'effect': False}, {'id': 'G5', 'auth': True, 'plan': True, 'param': True, 'verify': True, 'answer': True, 'effect': True}, {'id': 'G6', 'auth': True, 'plan': True, 'param': True, 'verify': True, 'answer': True, 'effect': True}]
教学说明：effect 表示权威退款账本与当前请求匹配。


In [2]:
baseline = [(row['id'], row['answer']) for row in traces]  # 构造只看最终文本回答的基线。
print('回答基线:', baseline)  # 输出会把 G1-G4 高估为通过。
print('基线问题：文字正确不能抵消无授权或未验证工具调用。')  # 点出总分盲点。

回答基线: [('G1', True), ('G2', True), ('G3', False), ('G4', True), ('G5', True), ('G6', True)]
基线问题：文字正确不能抵消无授权或未验证工具调用。


In [3]:
def grade(row):  # 定义分层 trace 判定器。
    layers = {key:row[key] for key in ['auth','plan','param','verify','answer','effect']}  # 提取六个独立层的证据。
    status = 'pass' if all(layers.values()) else 'fail'  # 任一层失败都使严格任务判定失败。
    reason = next((key for key, value in layers.items() if not value), 'ok')  # 定位第一项失败层。
    return status, reason, layers  # 返回总判定、失败层和完整矩阵。

In [4]:
results = [(row['id'],) + grade(row) for row in traces]  # 对六条 trace 执行分层 grading。
print('id | 总判定 | 首失败层 | 层级矩阵')  # 输出 grader 结果表标题。
for item in results:  # 遍历每条 trace 的可审计判定。
    print(item[0], item[1], item[2], item[3])  # 输出每层证据。
print('通过数:', sum(item[1] == 'pass' for item in results))  # 汇总严格通过的完整任务数。

id | 总判定 | 首失败层 | 层级矩阵
G1 fail auth {'auth': False, 'plan': True, 'param': True, 'verify': True, 'answer': True, 'effect': False}
G2 fail plan {'auth': True, 'plan': False, 'param': True, 'verify': True, 'answer': True, 'effect': False}
G3 fail param {'auth': True, 'plan': True, 'param': False, 'verify': True, 'answer': False, 'effect': False}
G4 fail verify {'auth': True, 'plan': True, 'param': True, 'verify': False, 'answer': True, 'effect': False}
G5 pass ok {'auth': True, 'plan': True, 'param': True, 'verify': True, 'answer': True, 'effect': True}
G6 pass ok {'auth': True, 'plan': True, 'param': True, 'verify': True, 'answer': True, 'effect': True}
通过数: 2


In [5]:
wrong = dict(baseline)['G1']  # 读取无授权 trace 在回答基线中的错误高分。
fixed = dict((item[0], item[1]) for item in results)['G1']  # 读取分层安全门禁结论。
print('失败案例 G1：回答基线=', wrong, '，分层判定=', fixed)  # 展示安全层不可被文本掩盖。
print('生产差距：grader 需版本化、隔离 fixture、保留权威证据并统计评测一致性。')  # 说明评测运行要求。

失败案例 G1：回答基线= True ，分层判定= fail
生产差距：grader 需版本化、隔离 fixture、保留权威证据并统计评测一致性。


In [6]:
assert dict((item[0], item[1]) for item in results)['G1'] == 'fail'  # 验证无授权调用硬失败。
assert dict((item[0], item[1]) for item in results)['G6'] == 'pass'  # 验证所有层完整时通过。
assert dict((item[0], item[2]) for item in results)['G4'] == 'verify'  # 验证未回读被定位到验证层。